In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/df_final.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (62728, 30)
Columns: ['timestamp', 'price', 'load', 'wind_offshore', 'wind_onshore', 'solar', 'hour', 'day_of_week', 'month', 'temperature', 'wind_speed', 'is_weekend', 'gas_price', 'coal_price', 'price_lag_24h', 'price_lag_168h', 'price_rolling_24h', 'price_rolling_168h', 'co2_price', 'is_holiday', 'is_hol_or_week', 'total_generation', 'net_export', 'coal_generation', 'gas_generation', 'nuclear_generation', 'actual_wind_offshore', 'actual_wind_onshore', 'actual_solar', 'actual_load']


In [3]:
# Cyclic transformation
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Check
print("New columns:")
new_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
print(df[new_cols].describe().round(3))

New columns:
        hour_sin   hour_cos    dow_sin    dow_cos  month_sin  month_cos
count  62728.000  62728.000  62728.000  62728.000  62728.000  62728.000
mean      -0.000     -0.000      0.001     -0.000      0.011      0.011
std        0.707      0.707      0.707      0.707      0.707      0.707
min       -1.000     -1.000     -0.975     -0.901     -1.000     -1.000
25%       -0.707     -0.707     -0.782     -0.901     -0.500     -0.500
50%        0.000     -0.000      0.000     -0.223      0.000      0.000
75%        0.707      0.707      0.782      0.623      0.866      0.866
max        1.000      1.000      0.975      1.000      1.000      1.000


In [4]:
# Lag features for fuel prices
df['gas_price_lag_24h'] = df['gas_price'].shift(24)
df['gas_price_lag_168h'] = df['gas_price'].shift(168)

df['coal_price_lag_24h'] = df['coal_price'].shift(24)
df['coal_price_lag_168h'] = df['coal_price'].shift(168)

df['co2_price_lag_24h'] = df['co2_price'].shift(24)

# Sanity check — first 170 rows should have NaN for 168h lags
print("NaNs from fuel lags:")
lag_cols = ['gas_price_lag_24h', 'gas_price_lag_168h',
            'coal_price_lag_24h', 'coal_price_lag_168h',
            'co2_price_lag_24h']
print(df[lag_cols].isnull().sum())

NaNs from fuel lags:
gas_price_lag_24h       24
gas_price_lag_168h     168
coal_price_lag_24h      24
coal_price_lag_168h    168
co2_price_lag_24h       24
dtype: int64


In [5]:
# Renewable share — ratio of renewables to total generation
# More informative than absolute MW values
df['renewable_share'] = (
    df['wind_onshore'] + df['wind_offshore'] + df['solar']
) / df['total_generation'].replace(0, np.nan)

# Fuel cost index — weighted combo of gas and coal
# Gas is the strongest single predictor from EDA
df['fuel_cost_index'] = df['gas_price'] * 0.6 + df['coal_price'] * 0.4

# Dispatchable generation — thermal + nuclear (non-renewable, price-setting)
df['dispatchable_gen'] = (
    df['coal_generation'] + df['gas_generation'] + df['nuclear_generation']
)

# Demand-supply balance — load minus total generation
# Negative = oversupply (pushes price down), positive = undersupply
df['demand_supply_gap'] = df['load'] - df['total_generation']

# Sanity check
check_cols = ['renewable_share', 'fuel_cost_index',
              'dispatchable_gen', 'demand_supply_gap']
print(df[check_cols].describe().round(2))

       renewable_share  fuel_cost_index  dispatchable_gen  demand_supply_gap
count         62728.00         62728.00          62728.00           62728.00
mean              0.37            75.87          24956.94            -651.71
std               0.19            57.27          10403.24            7389.39
min               0.01            17.59           4158.75          -23374.00
25%               0.21            36.35          16982.19           -6006.12
50%               0.36            61.80          24708.71            -553.75
75%               0.51            78.63          32196.06            4875.03
max               1.07           349.46          57427.75           20980.00


In [6]:
# Peak hour flag (morning 7-10, evening 17-21)
df['is_peak_hour'] = df['hour'].isin(range(7, 11)) | df['hour'].isin(range(17, 22))
df['is_peak_hour'] = df['is_peak_hour'].astype(int)

# Wind suppression during peak — high wind during peak hours
# This is when wind has biggest price impact (merit order effect)
df['wind_x_peak'] = (
    (df['wind_onshore'] + df['wind_offshore']) * df['is_peak_hour']
)

# Gas price pressure during peak demand
df['gas_x_peak'] = df['gas_price'] * df['is_peak_hour']

# Solar suppression — solar during high demand hours
df['solar_x_demand'] = df['solar'] * df['load']

# Renewable share during peak — captures merit order effect
df['renewable_share_x_peak'] = df['renewable_share'] * df['is_peak_hour']

# Sanity check
interact_cols = ['is_peak_hour', 'wind_x_peak', 'gas_x_peak',
                 'solar_x_demand', 'renewable_share_x_peak']
print(df[interact_cols].describe().round(2))
print(f"\nPeak hours: {df['is_peak_hour'].sum():,} rows ({df['is_peak_hour'].mean()*100:.1f}%)")

       is_peak_hour  wind_x_peak  gas_x_peak  solar_x_demand   
count      62728.00     62728.00    62728.00    6.272800e+04  \
mean           0.38      5530.37       16.84    3.611918e+08   
std            0.48      9768.79       34.72    5.625654e+08   
min            0.00         0.00        0.00    0.000000e+00   
25%            0.00         0.00        0.00    0.000000e+00   
50%            0.00         0.00        0.00    1.151225e+07   
75%            1.00      7630.31       24.76    5.707494e+08   
max            1.00     51550.50      339.20    3.067909e+09   

       renewable_share_x_peak  
count                62728.00  
mean                     0.13  
std                      0.20  
min                      0.00  
25%                      0.00  
50%                      0.00  
75%                      0.25  
max                      0.97  

Peak hours: 23,526 rows (37.5%)


In [7]:
# Energy crisis period flag (2021-2023 gas/energy crisis)
df['is_crisis_period'] = (
    (df['timestamp'] >= '2021-01-01') &
    (df['timestamp'] <= '2023-12-31')
).astype(int)

# High price regime — above 75th percentile historically
price_75 = df['price'].quantile(0.75)
df['is_high_price_regime'] = (df['price'] > price_75).astype(int)

# Negative price flag
df['is_negative_price'] = (df['price'] < 0).astype(int)

# Year feature — captures long-term trend (energy transition)
df['year'] = df['timestamp'].dt.year

# Sanity check
print(f"Crisis period rows:    {df['is_crisis_period'].sum():,} ({df['is_crisis_period'].mean()*100:.1f}%)")
print(f"High price regime:     {df['is_high_price_regime'].sum():,} ({df['is_high_price_regime'].mean()*100:.1f}%)")
print(f"Negative price rows:   {df['is_negative_price'].sum():,} ({df['is_negative_price'].mean()*100:.1f}%)")
print(f"Years in dataset:      {sorted(df['year'].unique())}")

Crisis period rows:    26,254 (41.9%)
High price regime:     15,681 (25.0%)
Negative price rows:   2,037 (3.2%)
Years in dataset:      [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [8]:
# Check all NaNs before dropping
print("NaNs per column before drop:")
nan_cols = df.columns[df.isnull().any()].tolist()
print(df[nan_cols].isnull().sum().sort_values(ascending=False))

# Drop first 168 rows — covers all lag-induced NaNs
df = df.dropna(subset=nan_cols).reset_index(drop=True)

# Verify
print(f"\nShape after drop: {df.shape}")
print(f"Remaining NaNs:   {df.isnull().sum().sum()}")
print(f"New date range:   {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

NaNs per column before drop:
gas_price_lag_168h     168
coal_price_lag_168h    168
gas_price_lag_24h       24
coal_price_lag_24h      24
co2_price_lag_24h       24
dtype: int64

Shape after drop: (62560, 54)
Remaining NaNs:   0
New date range:   2019-01-15 → 2026-03-05


In [9]:
EXCLUDE_COLS = [
    'timestamp',
    'price',
    'is_high_price_regime',
    'is_negative_price',
    'actual_wind_offshore',
    'actual_wind_onshore',
    'actual_solar',
    'actual_load',
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f"Total columns:   {df.shape[1]}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Excluded:        {len(EXCLUDE_COLS)}")
print(f"\nFeatures going into model:")
for col in feature_cols:
    print(f"  {col}")

""" 
    df.to_csv('../data/df_features.csv', index=False)
    print(f"Saved: df_features.csv  ({df.shape[0]:,} rows, {df.shape[1]} cols)")
 """

Total columns:   54
Feature columns: 46
Excluded:        8

Features going into model:
  load
  wind_offshore
  wind_onshore
  solar
  hour
  day_of_week
  month
  temperature
  wind_speed
  is_weekend
  gas_price
  coal_price
  price_lag_24h
  price_lag_168h
  price_rolling_24h
  price_rolling_168h
  co2_price
  is_holiday
  is_hol_or_week
  total_generation
  net_export
  coal_generation
  gas_generation
  nuclear_generation
  hour_sin
  hour_cos
  dow_sin
  dow_cos
  month_sin
  month_cos
  gas_price_lag_24h
  gas_price_lag_168h
  coal_price_lag_24h
  coal_price_lag_168h
  co2_price_lag_24h
  renewable_share
  fuel_cost_index
  dispatchable_gen
  demand_supply_gap
  is_peak_hour
  wind_x_peak
  gas_x_peak
  solar_x_demand
  renewable_share_x_peak
  is_crisis_period
  year


' \n    df.to_csv(\'../data/df_features.csv\', index=False)\n    print(f"Saved: df_features.csv  ({df.shape[0]:,} rows, {df.shape[1]} cols)")\n '

In [10]:
# Define features and target
EXCLUDE_COLS = [
    'timestamp',
    'price',
    'is_high_price_regime',
    'is_negative_price',
    'actual_wind_offshore',
    'actual_wind_onshore',
    'actual_solar',
    'actual_load',
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

X = df[feature_cols]
y = df['price']

# Chronological train/test split
split_date = '2025-01-01'

X_train = X[df['timestamp'] < split_date]
X_test  = X[df['timestamp'] >= split_date]
y_train = y[df['timestamp'] < split_date]
y_test  = y[df['timestamp'] >= split_date]

print(f"Features:      {X.shape[1]}")
print(f"Train size:    {len(X_train):,} rows  ({X_train.shape[0] / len(X) * 100:.1f}%)")
print(f"Test size:     {len(X_test):,} rows   ({X_test.shape[0] / len(X) * 100:.1f}%)")
print(f"Train range:   {df[df['timestamp'] < split_date]['timestamp'].min().date()} → {df[df['timestamp'] < split_date]['timestamp'].max().date()}")
print(f"Test range:    {df[df['timestamp'] >= split_date]['timestamp'].min().date()} → {df[df['timestamp'] >= split_date]['timestamp'].max().date()}")

Features:      46
Train size:    52,266 rows  (83.5%)
Test size:     10,294 rows   (16.5%)
Train range:   2019-01-15 → 2024-12-31
Test range:    2025-01-01 → 2026-03-05


In [11]:
def evaluate_model(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{name:<25} RMSE: {rmse:7.2f} €/MWh  |  MAE: {mae:7.2f} €/MWh  |  R²: {r2:.4f}")
    return {'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2}

# Store all results here
results = []

In [12]:
# --- Baseline 1: DummyRegressor ---
# models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

# preprocessing / pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline

# metrics (za evaluate_model funkciju)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.dummy import DummyRegressor
dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
results.append(evaluate_model('Dummy (mean)', y_test, y_pred_dummy))

# --- Baseline 2: Naive (price_lag_24h as prediction) ---
# No fitting needed — we just use the lag column directly
y_pred_naive = X_test['price_lag_24h'].values
results.append(evaluate_model('Naive (lag 24h)', y_test, y_pred_naive))

# --- Baseline 3: Linear Regression ---
lr_pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('model', LinearRegression())
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
results.append(evaluate_model('Linear Regression', y_test, y_pred_lr))

Dummy (mean)              RMSE:   50.56 €/MWh  |  MAE:   34.19 €/MWh  |  R²: -0.0082
Naive (lag 24h)           RMSE:   39.40 €/MWh  |  MAE:   25.34 €/MWh  |  R²: 0.3876
Linear Regression         RMSE:   24.32 €/MWh  |  MAE:   17.35 €/MWh  |  R²: 0.7666


In [13]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Random Forest — no GPU support, use all CPU cores
rf = Pipeline([
    ('scaler', RobustScaler()),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])
rf.fit(X_train, y_train)
results.append(evaluate_model('Random Forest', y_test, rf.predict(X_test)))

# XGBoost — GPU
xgb_model = Pipeline([
    ('scaler', RobustScaler()),
    ('model', XGBRegressor(
        n_estimators=300, random_state=42,
        device='cuda',
        verbosity=0
    ))
])
xgb_model.fit(X_train, y_train)
results.append(evaluate_model('XGBoost', y_test, xgb_model.predict(X_test)))

# LightGBM — GPU
lgbm_model = Pipeline([
    ('scaler', RobustScaler()),
    ('model', LGBMRegressor(
        n_estimators=300, random_state=42,
        device='gpu',
        verbose=-1
    ))
])
lgbm_model.fit(X_train, y_train)
results.append(evaluate_model('LightGBM', y_test, lgbm_model.predict(X_test)))

# CatBoost — GPU
cat_model = Pipeline([
    ('scaler', RobustScaler()),
    ('model', CatBoostRegressor(
        iterations=300, random_state=42,
        task_type='GPU',
        verbose=0
    ))
])
cat_model.fit(X_train, y_train)
results.append(evaluate_model('CatBoost', y_test, cat_model.predict(X_test)))

Random Forest             RMSE:   22.33 €/MWh  |  MAE:   14.59 €/MWh  |  R²: 0.8034
XGBoost                   RMSE:   23.74 €/MWh  |  MAE:   16.03 €/MWh  |  R²: 0.7776
LightGBM                  RMSE:   20.59 €/MWh  |  MAE:   13.89 €/MWh  |  R²: 0.8328
CatBoost                  RMSE:   20.71 €/MWh  |  MAE:   13.87 €/MWh  |  R²: 0.8309


In [14]:
!pip install xgboost lightgbm catboost

In [15]:
# Summary table
results_df = pd.DataFrame(results).sort_values('rmse')
print("\n=== MODEL COMPARISON ===")
print(results_df.to_string(index=False))


=== MODEL COMPARISON ===
            model      rmse       mae        r2
         LightGBM 20.589336 13.888789  0.832801
         CatBoost 20.705584 13.874581  0.830907
    Random Forest 22.328481 14.593080  0.803361
          XGBoost 23.744913 16.027583  0.777622
Linear Regression 24.323957 17.352679  0.766644
  Naive (lag 24h) 39.403971 25.344373  0.387607
     Dummy (mean) 50.559294 34.189661 -0.008213


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform, loguniform

tscv = TimeSeriesSplit(n_splits=5, gap=168)

param_dist = {
    'n_estimators': randint(300, 1200),
    'max_depth': randint(3, 10),
    'learning_rate': loguniform(0.005, 0.2),
    'num_leaves': randint(20, 120),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_samples': randint(10, 100),
    'reg_alpha': loguniform(1e-8, 10),
    'reg_lambda': loguniform(1e-8, 10)
}

model = LGBMRegressor(
    random_state=42,
    verbose=-1
)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=150,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f"\nBest RMSE (CV):  {-random_search.best_score_:.2f} €/MWh")
print(f"Best params:     {random_search.best_params_}")

Fitting 5 folds for each of 150 candidates, totalling 750 fits


In [ ]:
from sklearn.model_selection import GridSearchCV

# Focused grid around promising values from Random Search
param_grid = {
    'n_estimators': [200, 400, 600],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63, 127],
}

grid_search = GridSearchCV(
    estimator=LGBMRegressor(random_state=42, verbose=-1),
    param_grid=param_grid,
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best RMSE (CV): {-grid_search.best_score_:.2f} €/MWh")
print(f"Best params:    {grid_search.best_params_}")

In [ ]:
import optuna
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

tscv = TimeSeriesSplit(n_splits=5, gap=168)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1,
        # nema device='gpu' i nema n_jobs
    }

    model = LGBMRegressor(**params)
    scores = cross_val_score(
        model, X_train, y_train,
        cv=tscv,
        scoring='neg_root_mean_squared_error'
        # nema n_jobs
    )
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nBest RMSE (CV): {-study.best_value:.2f} €/MWh")
print(f"Best params:    {study.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]

Best trial: 12. Best value: -49.4302: 100%|██████████| 50/50 [11:38<00:00, 13.98s/it]


Best RMSE (CV): 49.43 €/MWh
Best params:    {'n_estimators': 966, 'max_depth': 3, 'learning_rate': 0.010725350049415877, 'num_leaves': 95, 'subsample': 0.6191392785818687, 'colsample_bytree': 0.759430406861425, 'reg_alpha': 0.0001950224783505618, 'reg_lambda': 1.8514326922496438e-07}


In [ ]:
import sys
!"{sys.executable}" -m pip install optuna


   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   -------------------- ------------------- 1/2 [optuna]
   ---------------------------------------- 2/2 [optuna]



In [ ]:
tscv_check = TimeSeriesSplit(n_splits=5, gap=168)
for fold, (train_idx, val_idx) in enumerate(tscv_check.split(X_train)):
    val_dates = df[df['timestamp'] < '2025-01-01']['timestamp'].iloc[val_idx]
    print(f"Fold {fold+1}: val {val_dates.min().date()} → {val_dates.max().date()}  ({len(val_idx):,} rows)")

Fold 1: val 2020-01-13 → 2021-01-09  (8,711 rows)
Fold 2: val 2021-01-10 → 2022-01-07  (8,711 rows)
Fold 3: val 2022-01-08 → 2023-01-05  (8,711 rows)
Fold 4: val 2023-01-06 → 2024-01-03  (8,711 rows)
Fold 5: val 2024-01-04 → 2024-12-31  (8,711 rows)


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
import numpy as np

# Use only last fold for CV — closest to test period
tscv_last = TimeSeriesSplit(n_splits=5, gap=168)

# Get last fold indices manually
splits = list(tscv_last.split(X_train))
last_train_idx, last_val_idx = splits[-1]

X_tr = X_train.iloc[last_train_idx]
y_tr = y_train.iloc[last_train_idx]
X_val = X_train.iloc[last_val_idx]
y_val = y_train.iloc[last_val_idx]

print(f"Val period: {df[df['timestamp'] < '2025-01-01']['timestamp'].iloc[last_val_idx].min().date()} → {df[df['timestamp'] < '2025-01-01']['timestamp'].iloc[last_val_idx].max().date()}")

# Quick check with default LightGBM on last fold
lgbm_check = LGBMRegressor(random_state=42, verbose=-1)
lgbm_check.fit(X_tr, y_tr)
y_pred_val = lgbm_check.predict(X_val)

from sklearn.metrics import mean_squared_error
rmse_val = mean_squared_error(y_val, y_pred_val, squared=False)
print(f"RMSE on last fold (2024): {rmse_val:.2f} €/MWh")

Val period: 2024-01-04 → 2024-12-31
RMSE on last fold (2024): 25.38 €/MWh


In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'verbose': -1,
    }

    model = LGBMRegressor(**params)
    
    # Use only last fold — closest to test period
    splits = list(TimeSeriesSplit(n_splits=5, gap=168).split(X_train))
    last_train_idx, last_val_idx = splits[-1]
    
    model.fit(X_train.iloc[last_train_idx], y_train.iloc[last_train_idx])
    y_pred = model.predict(X_train.iloc[last_val_idx])
    rmse = mean_squared_error(
        y_train.iloc[last_val_idx], y_pred, squared=False
    )
    return -rmse

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\nBest RMSE (val 2024): {-study.best_value:.2f} €/MWh")
print(f"Best params: {study.best_params}")

Best trial: 51. Best value: -24.0017: 100%|██████████| 100/100 [09:06<00:00,  5.47s/it]


Best RMSE (val 2024): 24.00 €/MWh
Best params: {'n_estimators': 526, 'max_depth': 10, 'learning_rate': 0.045740485763962564, 'num_leaves': 143, 'subsample': 0.7696015869299223, 'colsample_bytree': 0.7366989204188319, 'reg_alpha': 3.0689953785955126e-06, 'reg_lambda': 0.4009970321577672}


In [ ]:
# Fit final model with best params on entire train
final_lgbm = LGBMRegressor(
    **study.best_params,
    random_state=42,
    verbose=-1
)
final_lgbm.fit(X_train, y_train)

# Evaluate on test set
y_pred_tuned = final_lgbm.predict(X_test)
rmse_tuned = mean_squared_error(y_test, y_pred_tuned, squared=False)
mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

print(f"Default LightGBM:  RMSE: 20.59 €/MWh")
print(f"Tuned LightGBM:    RMSE: {rmse_tuned:.2f} €/MWh  |  MAE: {mae_tuned:.2f}  |  R²: {r2_tuned:.4f}")

Default LightGBM:  RMSE: 20.59 €/MWh
Tuned LightGBM:    RMSE: 20.93 €/MWh  |  MAE: 13.97  |  R²: 0.8272


In [ ]:
# Load one cross-border file
cross_check = pd.read_csv(
    '../data/Cross-border_physical_flows_202201010000_202501010000_Hour.csv', 
    sep=';'
)

# Pick one specific timestamp
ts = '2022-01-01 00:00:00'
row = cross_check[cross_check['Start date'] == '01.01.2022 00:00']

print("Raw Excel row:")
print(row.T)  # transpose za lakše čitanje

print("\nNaš df_final za isti timestamp:")
print(df[df['timestamp'] == ts][['timestamp', 'net_export']])

NameError: name 'pd' is not defined